# Heart Disease Classification — Feature Engineering & Preprocessing

## Objectives

The objective of this notebook is to address the data quality issues identified in EDA and prepare the dataset for modeling. This includes binarizing the target variable from `0-4` to `0/1`, mode imputation for missing values in `ca` and `thal`, grouping rare 
categorical categories, log transforming `oldpeak`, handling outliers, and scaling continuous features.

Given the small sample size of 303 patients, preserving as much data as possible is a priority — this motivated the choice of mode imputation over dropping rows for missing values. All transformations are applied after the train/test split to prevent data leakage, 
meaning imputation and scaling parameters are learned from the training set only and then applied to the test set.

## Output
- The output of this notebook is a cleaned, transformed, and scaled dataset saved to `data/processed/` and ready for modeling in `03_modeling.ipynb`.

## 2.1 Setup & Load Data

Importing the same core libraries as `01_eda.ipynb` with the addition of sklearn modules for train/test splitting, imputation, and scaling. Path constants and random state are defined here to ensure reproducibility and consistent file references throughout the notebook.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore') # suppress sklearn warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

sns.set_theme()
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_STATE = 42
RAW_DATA_PATH = '../data/raw/heart.csv'
PROCESSED_DATA_PATH = '../data/processed/heart_cleaned.csv'

Loading the raw dataset fresh from `data/raw/heart.csv` to ensure all transformations in this notebook start from the original unmodified data. Column names applied manually since the raw UCI file has no header row, consistent with `01_eda.ipynb`.

In [2]:
columns = ['age','sex','cp', 'trestbps','chol', 'fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','target']
df = pd.read_csv(RAW_DATA_PATH, names = columns, header = None)
print(df.shape)
df.head()

(303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 2.2 Target Binarization

Binarizing the target variable from its raw `0-4` encoding to binary `0/1`. Values `1-4` all indicate presence of heart disease at varying severity — for binary classification I only need to distinguish disease present vs not present. This decision was motivated by 
the consistent overlap between categories `1-4` observed in bivariate analysis.

In [3]:
df['target'] = (df['target']>0).astype(int)

In [4]:
df['target'].value_counts()

target
0    164
1    139
Name: count, dtype: int64

Target successfully binarized. 164 patients with no disease (`0`) and 139 with disease present (`1`), confirming the ~54/46 split identified in EDA.

## 2.3 Categorical Feature Engineering

In EDA I identified that `restecg`, `thal`, `slope`, and `ca` have rare categories that could cause instability during modeling. Grouping these reduces noise by ensuring the model has enough examples to learn reliable patterns from each category.

For `restecg`, category `1.0` is extremely rare (~1% of patients). Since both `1.0` and `2.0` represent abnormal ECG results, I combine them into a single abnormal category (`1.0`) and leave category `0.0` as normal.

For `thal`, category `6.0` (fixed defect) is very rare. Although fixed and reversible defects are clinically distinct, the sample size for category `6.0` is too small to be reliable, so I group it with category `7.0` (reversible defect).

For `slope`, category `3.0` is rare. Since category `2.0` (flat) had the most disease cases in bivariate analysis and is the most informative, I group upsloping (`1.0`) and downsloping (`3.0`) together and leave flat (`2.0`) as is.

For `ca`, categories `1.0`-`3.0` all represent one or more blocked vessels. Given the rarity of `ca=3.0`, I combine `1.0`, `2.0`, and `3.0` into a single category (`1.0`) representing any blocked vessels, and leave `0.0` as no blocked vessels. This also creates a clinically meaningful binary question: are any vessels blocked?

In [4]:
df['restecg'] = df['restecg'].map({0.0: 0.0, 1.0: 1.0, 2.0: 1.0})
df['thal'] = df['thal'].map({'3.0': 3.0, '6.0': 7.0, '7.0': 7.0, '?': np.nan})
df['slope'] = df['slope'].map({1.0: 1.0, 2.0: 2.0, 3.0: 1.0})
df['ca'] = df['ca'].map({'0.0': 0.0, '1.0': 1.0, '2.0': 1.0, '3.0': 1.0, '?': np.nan})

In [7]:
print(df['restecg'].value_counts())
print(df['thal'].value_counts())
print(df['slope'].value_counts())
print(df['ca'].value_counts())

restecg
1.0    152
0.0    151
Name: count, dtype: int64
thal
3.0    166
7.0    135
Name: count, dtype: int64
slope
1.0    163
2.0    140
Name: count, dtype: int64
ca
0.0    176
1.0    123
Name: count, dtype: int64


Category groupings confirmed. Missing values in `ca` (4) and `thal` (2) preserved as `NaN` for imputation in section 2.5. `Restecg` and `slope` show no missing values as expected.

## 2.4 Train/Test Split

Before imputing and scaling, I split the data into training and test sets to avoid data leakage. The split follows a standard 80/20 ratio — 242 patients for training and 61 for testing. By splitting first, all transformation parameters such as imputation values and 
scaling statistics are learned from the training set only and then applied to both sets. This ensures the test set remains a true representation of unseen data and the model is evaluated fairly.

In [5]:
X = df.drop('target', axis = 1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_STATE)

In [14]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(242, 13)
(61, 13)
(242,)
(61,)


The split produced 242 training samples and 61 test samples, confirming the 80/20 ratio. 
`X_train` and `X_test` both have 13 features as expected.

## 2.5 Missing Value Imputation

`Ca` and `thal` contained missing values encoded as `?` which were preserved as `NaN` through the pipeline. I use mode imputation via `SimpleImputer` fitted on training data only, then applied to both train and test sets. Mode imputation was chosen over dropping rows to conserve data given the small sample size of 303 patients, and over mean imputation since `ca` and `thal` are categorical variables where a mean value would not represent a valid category.

In [6]:
imputer = SimpleImputer(strategy = 'most_frequent')
imputer.fit(X_train[['ca', 'thal']])
X_train[['ca', 'thal']] = imputer.transform(X_train[['ca', 'thal']])
X_test[['ca', 'thal']] = imputer.transform(X_test[['ca', 'thal']])

In [16]:
print(X_train[['ca', 'thal']].isnull().sum())
print(X_test[['ca', 'thal']].isnull().sum())

ca      0
thal    0
dtype: int64
ca      0
thal    0
dtype: int64


## 2.6 Outlier Handling

From EDA, `chol` and `trestbps` were flagged as having extreme outliers. I handle these using capping at the 99th percentile rather than removal, since the small sample size of 303 patients makes preserving every row a priority. Capping was chosen over keeping the 
values as-is because extreme values distort `StandardScaler` by inflating the mean and standard deviation, compressing the rest of the distribution during scaling. At the 99th percentile only roughly the top 3 values in the dataset are affected, making the impact 
on the overall distribution minimal while preventing scaling distortion. The 99th percentile thresholds were calculated from the training set only and applied to both train and test sets to prevent data leakage.

In [7]:
chol_cap = X_train['chol'].quantile(0.99)
trestbps_cap = X_train['trestbps'].quantile(0.99)

X_train['chol'] = X_train['chol'].clip(upper=chol_cap)
X_test['chol'] = X_test['chol'].clip(upper=chol_cap)

X_train['trestbps'] = X_train['trestbps'].clip(upper=trestbps_cap)
X_test['trestbps'] = X_test['trestbps'].clip(upper=trestbps_cap)

In [18]:
print(X_train['chol'].max())
print(X_train['trestbps'].max())

401.6700000000001
180.0


`chol` capped at 401.67, clinically represents severe hypercholesterolemia, within the range of documented cases and more defensible than the original extreme of 564. 

`trestbps` capped at 180 mmHg, classified as hypertensive crisis territory clinically but a documented real value, more reasonable than the original 200. Both cap values are physiologically plausible, meaning the outliers are limited to realistic thresholds rather 
than being removed or artificially invented. 

## 2.7 Log Transform

In EDA, `oldpeak` was identified as heavily right skewed with values concentrated near 0. Skewed features can distort distance-based models and affect scaling, so I apply a log 
transformation to compress the long right tail. However, since `oldpeak` contains values at exactly 0, a standard `log(x)` transformation is undefined at 0. Instead I use `np.log1p()` which computes `log(x + 1)`, shifting all values by 1 before taking the 
log and making it safe for zero values.

In [8]:
X_train['oldpeak'] = np.log1p(X_train['oldpeak'])
X_test['oldpeak'] = np.log1p(X_test['oldpeak'])

In [9]:
print(X_train['oldpeak'].min())
print(X_train['oldpeak'].max())

0.0
1.9740810260220096


`min = 0.0` confirms that patients with `oldpeak = 0` correctly map to 0 after `log1p` transformation. `max = 1.97` confirms the original max of 6.2 has been compressed to 1.97, indicating the long right tail has been significantly reduced. The log transform 
worked as expected.

## 2.8 Feature Scaling

To ensure continuous features are on a comparable scale before modeling, I apply `StandardScaler` to the 5 continuous features: `age`, `trestbps`, `chol`, `thalach`, and `oldpeak`. `StandardScaler` transforms each feature to have mean = 0 and std = 1 using the formula `z = (x - mean) / std`. Values below the mean become negative and values above become positive. Categorical features do not need scaling since they represent discrete labels, not magnitudes. The scaler is fit on training data only and applied to both train and test sets to prevent data leakage.

In [10]:
scaler = StandardScaler()
scaler.fit(X_train[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']])
X_train[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']] = scaler.transform(X_train[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']])
X_test[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']] = scaler.transform(X_test[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']])

In [11]:
print(X_train['age'].mean())
print(X_train['age'].std())

2.5691111313639987e-17
1.0020725410834093


Verified scaling on `age` — mean ≈ 0 and std ≈ 1 confirm `StandardScaler` was applied correctly. The small deviation from exactly 0 and 1 is due to floating point rounding error and can be ignored.

## 2.9 Save Processed Data

Now that feature engineering is complete, I save the processed data to `data/processed/`. Features are saved as `X_train.csv` and `X_test.csv`, targets as `y_train.csv` and `y_test.csv`. These four files will be loaded directly in `03_modeling.ipynb`.

In [12]:
X_train.to_csv('../data/processed/X_train.csv', index = False)
X_test.to_csv('../data/processed/X_test.csv', index = False)
y_train.to_csv('../data/processed/y_train.csv', index = False)
y_test.to_csv('../data/processed/y_test.csv', index = False)